In [ ]:
import os
import pandas as pd

input_dir = r"../data/gimme_results_final"
output_dir = r"../data/gimme_results_final_csv"

os.makedirs(output_dir, exist_ok=True)

for file in os.listdir(input_dir):
    if file.endswith(".xlsx"):
        in_path = os.path.join(input_dir, file)
        out_name = file.replace(".xlsx", ".csv")
        out_path = os.path.join(output_dir, out_name)

        df = pd.read_excel(in_path)
        df.to_csv(out_path, index=False)

print("✅ All xlsx converted to csv")

In [ ]:
import pandas as pd

# Raw Excel file path
excel_file = r"../data/label-2.xlsx"

# Read Excel
df = pd.read_excel(excel_file)

# Save as a GBK-encoded CSV (for Chinese Windows compatibility)
csv_file = excel_file.replace('.xlsx', '.csv')
df.to_csv(csv_file, index=False, encoding='gbk')

print(f"✅ 已成功将 {excel_file} 转换为 GBK 编码的 CSV 文件：{csv_file}")

In [ ]:
import os
import re
import shutil

# ====== Config paths ======
COMBO_DIR = r"../data/gimme_results_final_csv"
H2O2_BACKUP_DIR = COMBO_DIR + "_h2o2_backup"  # create the backup directory

# Auto-create a backup directory
os.makedirs(H2O2_BACKUP_DIR, exist_ok=True)

def clean_drug_name(name: str) -> str:
    """提取干净药物名：取第一个 '-' 前部分，转大写，只保留字母数字"""
    name = name.strip().upper()
    base = name.split('-')[0].strip()
    base = re.sub(r'[^A-Z0-9]', '', base)  # e.g. "H2O2" -> "H2O2", "ampicillin" -> "AMPICILLIN"
    return base

def is_h2o2_like(name: str) -> bool:
    """判断是否为 H2O2 或类似对照（不区分大小写）"""
    clean = name.strip().lower()
    return 'h2o2' in clean or clean in ['control', 'water', 'ctrl']

def parse_combo(filename: str):
    """
    解析文件名，返回 (drug1_raw, drug2_raw) 或 None
    """
    if not filename.endswith('.csv') or not filename.startswith('gimme_flux_'):
        return None

    core = filename.replace('gimme_flux_', '').replace('.csv', '')

    if '__VS__' in core:
        parts = core.split('__VS__', 1)
        if len(parts) == 2:
            return parts[0], parts[1]
    elif '__' in core:
        parts = core.split('__', 1)
        if len(parts) == 2:
            return parts[0], parts[1]
    return None

# ====== Scan and classify files ======
all_files = [f for f in os.listdir(COMBO_DIR) if f.endswith('.csv')]
to_rename = []      # (old_name, new_name)
to_backup = []      # old_name (contains h2o2)

print("🔍 分析文件...")

for f in all_files:
    parsed = parse_combo(f)
    if not parsed:
        print(f"⚠️ 无法解析格式，跳过: {f}")
        continue

    raw1, raw2 = parsed
    d1_clean = clean_drug_name(raw1)
    d2_clean = clean_drug_name(raw2)

    # Check whether the name contains H2O2 (raw or cleaned)
    if is_h2o2_like(raw1) or is_h2o2_like(raw2) or d1_clean == 'H2O2' or d2_clean == 'H2O2':
        to_backup.append(f)
        continue

    # Skip invalid drug names
    if not d1_clean or not d2_clean:
        print(f"⚠️ 药物名无效，跳过: {f} → ({d1_clean}, {d2_clean})")
        continue

    new_name = f"gimme_flux_{d1_clean}-{d2_clean}.csv"
    if new_name != f:
        to_rename.append((f, new_name))
    else:
        # Already valid but a real two-drug combo: keep in place (no move)
        pass

# ====== Preview results ======
print(f"\n✅ 将重命名 {len(to_rename)} 个双药组合文件:")
for old, new in to_rename:
    print(f"  {old} → {new}")

print(f"\n📁 将移动 {len(to_backup)} 个含 H2O2/对照的文件到:")
print(f"   {H2O2_BACKUP_DIR}")
for f in to_backup[:5]:  # show only the first 5
    print(f"  {f}")
if len(to_backup) > 5:
    print(f"  ... 还有 {len(to_backup)-5} 个")

# ====== Confirm and execute ======
confirm = input("\n是否执行操作? 输入 'yes' 确认: ").strip().lower()
if confirm != 'yes':
    print("❌ 操作已取消。")
    exit()

# 1. Move H2O2 files first (avoid rename conflicts)
print("\n🔄 正在移动 H2O2 文件...")
for f in to_backup:
    src = os.path.join(COMBO_DIR, f)
    dst = os.path.join(H2O2_BACKUP_DIR, f)
    shutil.move(src, dst)
    print(f"  移动: {f}")

# 2. Rename two-drug files
print("\n🔄 正在重命名双药组合文件...")
for old_name, new_name in to_rename:
    old_path = os.path.join(COMBO_DIR, old_name)
    new_path = os.path.join(COMBO_DIR, new_name)

    # Prevent overwriting
    if os.path.exists(new_path):
        print(f"⚠️ 目标已存在，跳过: {new_name}")
        continue

    os.rename(old_path, new_path)
    print(f"  重命名: {old_name} → {new_name}")

print("\n✅ 所有操作完成！")
print(f"   - 双药组合已标准化，位于: {COMBO_DIR}")
print(f"   - H2O2/对照文件已备份至: {H2O2_BACKUP_DIR}")

In [1]:
import os
import pandas as pd
import numpy as np

# =========================
# Paths
# =========================
SINGLE_DIR = r"../data/imat-standard-output_gimme_csv"
COMBO_DIR  = r"../data/gimme_results_final_csv"
LABEL_FILE = r"../data/label-2.csv"
OUT_FILE   = r"../data/final_75_features.csv"

EPS = 1e-6

# =========================
# Utility functions
# =========================
def discretize(x):
    if abs(x) < EPS:
        return 0
    elif x > 0:
        return 1
    else:
        return -1


def normalize_name(name):
    return name.strip().upper()


def make_combo_key(a, b):
    a, b = normalize_name(a), normalize_name(b)
    return "-".join(sorted([a, b]))


def load_flux_states(fp):
    if not os.path.exists(fp):
        return {}

    df = pd.read_csv(fp)

    if "Solution_Status" in df.columns:
        df = df[df["Solution_Status"] == "optimal"]

    df["Reaction"] = df["Reaction"].astype(str)
    df["ΔFlux"] = pd.to_numeric(df["ΔFlux"], errors="coerce").fillna(0.0)
    df["state"] = df["ΔFlux"].apply(discretize)

    return dict(zip(df["Reaction"], df["state"]))


# =========================
# Parsing function (crash-proof)
# =========================
def parse_combo_name(filename):
    name = filename.replace("gimme_flux_", "").replace(".csv", "")

    # Must contain __VS__
    if "__VS__" not in name:
        return None, None

    parts = name.split("__VS__")

    # Guard against malformed formats
    if len(parts) != 2:
        return None, None

    partA, partB = parts

    try:
        A = partA.split("-")[0].strip()
        B = partB.split("-")[0].strip()
    except:
        return None, None

    return normalize_name(A), normalize_name(B)


# =========================
# 1. Read combination files (filter invalid)
# =========================
all_files = os.listdir(COMBO_DIR)

combo_files = []
invalid_files = []

for f in all_files:
    if not f.endswith(".csv"):
        continue

    if "__VS__" not in f:
        invalid_files.append(f)
        continue

    combo_files.append(f)

print("✅ Valid combo files:", len(combo_files))
print("⚠️ Invalid files skipped:", len(invalid_files))

if len(invalid_files) > 0:
    print("Example invalid files:", invalid_files[:5])


# =========================
# 2. Extract drug combinations
# =========================
combos = []
drug_set = set()

for f in combo_files:
    A, B = parse_combo_name(f)

    if A is None:
        print("⚠️ Skip parsing error:", f)
        continue

    combos.append((f, A, B))
    drug_set.update([A, B])

print("Total combos used:", len(combos))


# =========================
# 3. Single drugs
# =========================
single_state = {}
all_reactions = set()

for d in drug_set:
    fp = os.path.join(SINGLE_DIR, f"gimme_flux_{d}.csv")
    state = load_flux_states(fp)
    single_state[d] = state
    all_reactions.update(state.keys())

# Fix the reaction order (core)
all_reactions = sorted(list(all_reactions))


# =========================
# 4. Build features
# =========================
rows = []

for file_name, A, B in combos:
    print("Processing:", file_name)

    combo_key = make_combo_key(A, B)
    row = {"Combination": combo_key}

    A_state = single_state.get(A, {})
    B_state = single_state.get(B, {})
    C_state = load_flux_states(os.path.join(COMBO_DIR, file_name))

    for rxn in all_reactions:
        a = A_state.get(rxn, 0)
        b = B_state.get(rxn, 0)
        c = C_state.get(rxn, 0)

        # Baseline state
        row[f"{rxn}_A"] = a
        row[f"{rxn}_B"] = b
        row[f"{rxn}_C"] = c

        # Core features (fixed: discrete)
        row[f"{rxn}_A2C"] = discretize(c - a)
        row[f"{rxn}_B2C"] = discretize(c - b)

    rows.append(row)


# =========================
# 5. Handle labels
# =========================
label_df = pd.read_csv(LABEL_FILE, encoding="gbk")

label_df["Drug 1"] = label_df["Drug 1"].apply(normalize_name)
label_df["Drug 2"] = label_df["Drug 2"].apply(normalize_name)

label_df["Combination"] = label_df.apply(
    lambda x: make_combo_key(x["Drug 1"], x["Drug 2"]),
    axis=1
)

label_df = label_df.rename(
    columns={"Experimental Interaction Score": "Label"}
)

label_df = label_df[["Combination", "Label"]]


# =========================
# 6. Merge
# =========================
out_df = pd.DataFrame(rows)
out_df = out_df.merge(label_df, on="Combination", how="left")


# =========================
# 7. Save
# =========================
out_df.to_csv(OUT_FILE, index=False)

print("\n✅ Saved:", OUT_FILE)
print("Shape:", out_df.shape)
print("Missing labels:", out_df["Label"].isna().sum())

✅ Valid combo files: 0
⚠️ Invalid files skipped: 58
Example invalid files: ['gimme_flux_AMIKACIN-CEFOXITIN.csv', 'gimme_flux_AMIKACIN-GENTAMICIN.csv', 'gimme_flux_AMIKACIN-TETRACYCLINE.csv', 'gimme_flux_AMIKACIN-VANCOMYCIN.csv', 'gimme_flux_CEFOXITIN-CHLORAMPHENICOL.csv']
Total combos used: 0


KeyError: 'Combination'

In [4]:
import os
import pandas as pd
import numpy as np

# =========================
# Paths
# =========================
SINGLE_DIR = r"../data/imat-standard-output_gimme_csv"
COMBO_DIR  = r"../data/gimme_results_final_csv"
LABEL_FILE = r"../data/label-2.csv"
OUT_FILE   = r"../data/final_75_features.csv"

# More robust threshold (larger than before)
TH = 1e-4

# =========================
# Utility functions
# =========================
def discretize(x):
    if x > TH:
        return 1
    elif x < -TH:
        return -1
    else:
        return 0


def normalize_name(name):
    return name.strip().upper()


def make_combo_key(a, b):
    a, b = normalize_name(a), normalize_name(b)
    return "-".join(sorted([a, b]))


def load_flux_states(fp):
    if not os.path.exists(fp):
        return {}

    df = pd.read_csv(fp)

    if "Solution_Status" in df.columns:
        df = df[df["Solution_Status"] == "optimal"]

    df["Reaction"] = df["Reaction"].astype(str)
    df["ΔFlux"] = pd.to_numeric(df["ΔFlux"], errors="coerce").fillna(0.0)

    # Discretization (more robust)
    df["state"] = df["ΔFlux"].apply(discretize)

    return dict(zip(df["Reaction"], df["state"]))


# =========================
# Parsing function
# =========================
# def parse_combo_name(filename):
#     name = filename.replace("gimme_flux_", "").replace(".csv", "")

#     if "__VS__" not in name:
#         return None, None

#     parts = name.split("__VS__")
#     if len(parts) != 2:
#         return None, None

#     partA, partB = parts

#     try:
#         A = partA.split("-")[0].strip()
#         B = partB.split("-")[0].strip()
#     except:
#         return None, None

#     return normalize_name(A), normalize_name(B)
def parse_combo_name(filename):
    name = filename.replace("gimme_flux_", "").replace(".csv", "")

    # Split directly on "-"
    parts = name.split("-")

    if len(parts) != 2:
        return None, None

    A, B = parts

    return normalize_name(A), normalize_name(B)


# =========================
# 1. Read combination files
# =========================
all_files = os.listdir(COMBO_DIR)

combo_files = []
invalid_files = []

for f in all_files:
    if not f.endswith(".csv"):
        continue
    if "-" not in f:
        invalid_files.append(f)
        continue
    # if "__VS__" not in f:
    #     invalid_files.append(f)
    #     continue

    combo_files.append(f)

print("✅ Valid combo files:", len(combo_files))
print("⚠️ Invalid files skipped:", len(invalid_files))

if len(invalid_files) > 0:
    print("Example invalid files:", invalid_files[:5])


# =========================
# 2. Extract combinations
# =========================
combos = []
drug_set = set()

for f in combo_files:
    A, B = parse_combo_name(f)

    if A is None:
        print("⚠️ Skip parsing error:", f)
        continue

    combos.append((f, A, B))
    drug_set.update([A, B])

print("Total combos used:", len(combos))


# =========================
# 3. Single-drug state
# =========================
single_state = {}
all_reactions = set()

for d in drug_set:
    fp = os.path.join(SINGLE_DIR, f"gimme_flux_{d}.csv")
    state = load_flux_states(fp)
    single_state[d] = state
    all_reactions.update(state.keys())


# =========================
# Key change: add the combo reaction
# =========================
combo_state_cache = {}

for f, _, _ in combos:
    fp = os.path.join(COMBO_DIR, f)
    state = load_flux_states(fp)
    combo_state_cache[f] = state
    all_reactions.update(state.keys())


# Fix the reaction order
all_reactions = sorted(list(all_reactions))


# =========================
# 4. Build features
# =========================
rows = []

for file_name, A, B in combos:
    print("Processing:", file_name)

    combo_key = make_combo_key(A, B)
    row = {"Combination": combo_key}

    A_state = single_state.get(A, {})
    B_state = single_state.get(B, {})
    C_state = combo_state_cache[file_name]  # use the cache

    for rxn in all_reactions:
        a = A_state.get(rxn, 0)
        b = B_state.get(rxn, 0)
        c = C_state.get(rxn, 0)

        # ===== Baseline state =====
        row[f"{rxn}_A"] = a
        row[f"{rxn}_B"] = b
        row[f"{rxn}_C"] = c

        # ===== Optimization: direction-change features =====
        # A → C
        row[f"{rxn}_A_up"] = int(c > a)
        row[f"{rxn}_A_down"] = int(c < a)

        # B → C
        row[f"{rxn}_B_up"] = int(c > b)
        row[f"{rxn}_B_down"] = int(c < b)

    rows.append(row)


# =========================
# 5. Handle labels
# =========================
label_df = pd.read_csv(LABEL_FILE, encoding="gbk")

label_df["Drug 1"] = label_df["Drug 1"].apply(normalize_name)
label_df["Drug 2"] = label_df["Drug 2"].apply(normalize_name)

label_df["Combination"] = label_df.apply(
    lambda x: make_combo_key(x["Drug 1"], x["Drug 2"]),
    axis=1
)

label_df = label_df.rename(
    columns={"Experimental Interaction Score": "Label"}
)

label_df = label_df[["Combination", "Label"]]


# =========================
# 6. Merge
# =========================
out_df = pd.DataFrame(rows)
out_df = out_df.merge(label_df, on="Combination", how="left")


# =========================
# 7. Check labels
# =========================
missing = out_df["Label"].isna().sum()
print("\nMissing labels:", missing)

if missing > 0:
    print(out_df[out_df["Label"].isna()]["Combination"].head())


# =========================
# 8. Save
# =========================
out_df.to_csv(OUT_FILE, index=False)

print("\n✅ Saved:", OUT_FILE)
print("Shape:", out_df.shape)

✅ Valid combo files: 58
⚠️ Invalid files skipped: 0
Total combos used: 58
Processing: gimme_flux_AMIKACIN-CEFOXITIN.csv
Processing: gimme_flux_AMIKACIN-GENTAMICIN.csv
Processing: gimme_flux_AMIKACIN-TETRACYCLINE.csv
Processing: gimme_flux_AMIKACIN-VANCOMYCIN.csv
Processing: gimme_flux_CEFOXITIN-CHLORAMPHENICOL.csv
Processing: gimme_flux_CEFOXITIN-CIPROFLOXACIN.csv
Processing: gimme_flux_CEFOXITIN-CLARITHROMYCIN.csv
Processing: gimme_flux_CEFOXITIN-ERYTHROMYCIN.csv
Processing: gimme_flux_CEFOXITIN-GENTAMICIN.csv
Processing: gimme_flux_CEFOXITIN-LEVOFLOXACIN.csv
Processing: gimme_flux_CEFOXITIN-NALIDIXICACID.csv
Processing: gimme_flux_CEFOXITIN-NITROFURANTOIN.csv
Processing: gimme_flux_CEFOXITIN-OXACILLIN.csv
Processing: gimme_flux_CEFOXITIN-RIFAMPICIN.csv
Processing: gimme_flux_CEFOXITIN-SPECTINOMYCIN.csv
Processing: gimme_flux_CEFOXITIN-TETRACYCLINE.csv
Processing: gimme_flux_CEFOXITIN-TOBRAMYCIN.csv
Processing: gimme_flux_CEFOXITIN-TRIMETHOPRIM.csv
Processing: gimme_flux_CEFOXITIN-VAN

In [ ]:
# import os
# import pandas as pd
# import numpy as np

# # =========================
# # 路径
# # =========================
# SINGLE_DIR = r"../data/imat-standard-output_gimme_csv"
# COMBO_DIR  = r"../data/gimme_results_final_csv"
# LABEL_FILE = r"../data/label-2.csv"
# OUT_FILE   = r"../data/final_75_features.csv"

# TH = 1e-4

# # =========================
# # ⭐ 终极标准化函数（核心）
# # =========================
# def normalize_name(name):
#     name = str(name).strip().upper()
    
#     # 去掉所有可能导致不一致的符号
#     name = name.replace(" ", "")
#     name = name.replace("_", "")
#     name = name.replace("-", "")
    
#     return name


# def make_combo_key(a, b):
#     a, b = normalize_name(a), normalize_name(b)
#     return "-".join(sorted([a, b]))


# def discretize(x):
#     if x > TH:
#         return 1
#     elif x < -TH:
#         return -1
#     else:
#         return 0


# def load_flux_states(fp):
#     if not os.path.exists(fp):
#         return {}

#     df = pd.read_csv(fp)

#     if "Solution_Status" in df.columns:
#         df = df[df["Solution_Status"] == "optimal"]

#     df["Reaction"] = df["Reaction"].astype(str)
#     df["ΔFlux"] = pd.to_numeric(df["ΔFlux"], errors="coerce").fillna(0.0)
#     df["state"] = df["ΔFlux"].apply(discretize)

#     return dict(zip(df["Reaction"], df["state"]))


# # =========================
# # ⭐ 兼容所有文件名格式
# # =========================
# def parse_combo_name(filename):
#     name = filename.replace("gimme_flux_", "").replace(".csv", "")

#     if "__VS__" in name:
#         parts = name.split("__VS__")
#     else:
#         parts = name.split("-")

#     if len(parts) != 2:
#         return None, None

#     A, B = parts
#     return normalize_name(A), normalize_name(B)


# # =========================
# # 1. 读取组合文件
# # =========================
# all_files = os.listdir(COMBO_DIR)

# combo_files = []
# invalid_files = []

# for f in all_files:
#     if not f.endswith(".csv"):
#         continue

#     if "-" not in f and "__VS__" not in f:
#         invalid_files.append(f)
#         continue

#     combo_files.append(f)

# print("✅ Valid combo files:", len(combo_files))
# print("⚠️ Invalid files skipped:", len(invalid_files))


# # =========================
# # 2. 提取组合
# # =========================
# combos = []
# drug_set = set()

# for f in combo_files:
#     A, B = parse_combo_name(f)

#     if A is None:
#         print("⚠️ Skip:", f)
#         continue

#     combos.append((f, A, B))
#     drug_set.update([A, B])

# print("Total combos:", len(combos))


# # =========================
# # 3. 单药
# # =========================
# single_state = {}
# all_reactions = set()

# for d in drug_set:
#     fp = os.path.join(SINGLE_DIR, f"gimme_flux_{d}.csv")
#     state = load_flux_states(fp)
#     single_state[d] = state
#     all_reactions.update(state.keys())


# # =========================
# # 4. combo（含缓存 + reaction补全）
# # =========================
# combo_state_cache = {}

# for f, _, _ in combos:
#     fp = os.path.join(COMBO_DIR, f)
#     state = load_flux_states(fp)
#     combo_state_cache[f] = state
#     all_reactions.update(state.keys())

# all_reactions = sorted(list(all_reactions))


# # =========================
# # 5. 构建特征
# # =========================
# rows = []

# for file_name, A, B in combos:
#     print("Processing:", file_name)

#     combo_key = make_combo_key(A, B)
#     row = {"Combination": combo_key}

#     A_state = single_state.get(A, {})
#     B_state = single_state.get(B, {})
#     C_state = combo_state_cache[file_name]

#     for rxn in all_reactions:
#         a = A_state.get(rxn, 0)
#         b = B_state.get(rxn, 0)
#         c = C_state.get(rxn, 0)

#         # 基础
#         row[f"{rxn}_A"] = a
#         row[f"{rxn}_B"] = b
#         row[f"{rxn}_C"] = c

#         # ⭐ 优化变化特征
#         row[f"{rxn}_A_up"] = int(c > a)
#         row[f"{rxn}_A_down"] = int(c < a)

#         row[f"{rxn}_B_up"] = int(c > b)
#         row[f"{rxn}_B_down"] = int(c < b)

#     rows.append(row)


# # =========================
# # 6. label处理（自动对齐）
# # =========================
# label_df = pd.read_csv(LABEL_FILE, encoding="gbk")

# label_df["Drug 1"] = label_df["Drug 1"].apply(normalize_name)
# label_df["Drug 2"] = label_df["Drug 2"].apply(normalize_name)

# label_df["Combination"] = label_df.apply(
#     lambda x: make_combo_key(x["Drug 1"], x["Drug 2"]),
#     axis=1
# )

# label_df = label_df.rename(
#     columns={"Experimental Interaction Score": "Label"}
# )

# label_df = label_df[["Combination", "Label"]]


# # =========================
# # 7. 合并
# # =========================
# out_df = pd.DataFrame(rows)
# out_df = out_df.merge(label_df, on="Combination", how="left")


# # =========================
# # 8. 检查
# # =========================
# missing = out_df["Label"].isna().sum()
# print("\nMissing labels:", missing)

# if missing > 0:
#     print(out_df[out_df["Label"].isna()]["Combination"])


# # =========================
# # 9. 保存
# # =========================
# out_df.to_csv(OUT_FILE, index=False)

# print("\n✅ Saved:", OUT_FILE)
# print("Shape:", out_df.shape)

✅ Valid combo files: 58
⚠️ Invalid files skipped: 0
Total combos: 58
Processing: gimme_flux_AMIKACIN-CEFOXITIN.csv
Processing: gimme_flux_AMIKACIN-GENTAMICIN.csv
Processing: gimme_flux_AMIKACIN-TETRACYCLINE.csv
Processing: gimme_flux_AMIKACIN-VANCOMYCIN.csv
Processing: gimme_flux_CEFOXITIN-CHLORAMPHENICOL.csv
Processing: gimme_flux_CEFOXITIN-CIPROFLOXACIN.csv
Processing: gimme_flux_CEFOXITIN-CLARITHROMYCIN.csv
Processing: gimme_flux_CEFOXITIN-ERYTHROMYCIN.csv
Processing: gimme_flux_CEFOXITIN-GENTAMICIN.csv
Processing: gimme_flux_CEFOXITIN-LEVOFLOXACIN.csv
Processing: gimme_flux_CEFOXITIN-NALIDIXICACID.csv
Processing: gimme_flux_CEFOXITIN-NITROFURANTOIN.csv
Processing: gimme_flux_CEFOXITIN-OXACILLIN.csv
Processing: gimme_flux_CEFOXITIN-RIFAMPICIN.csv
Processing: gimme_flux_CEFOXITIN-SPECTINOMYCIN.csv
Processing: gimme_flux_CEFOXITIN-TETRACYCLINE.csv
Processing: gimme_flux_CEFOXITIN-TOBRAMYCIN.csv
Processing: gimme_flux_CEFOXITIN-TRIMETHOPRIM.csv
Processing: gimme_flux_CEFOXITIN-VANCOMYC

In [ ]:
import os
import pandas as pd
import numpy as np

# =========================
# Paths
# =========================
SINGLE_DIR = r"../data/imat-standard-output_gimme_csv"
COMBO_DIR  = r"../data/gimme_results_final_csv"
LABEL_FILE = r"../data/label-2.csv"
OUT_FILE   = r"../data/new-features.csv"

TH = 1e-4

# =========================
# Ultimate normalization function (core)
# =========================
def normalize_name(name):
    name = str(name).strip().upper()
    name = name.replace(" ", "").replace("_", "").replace("-", "")
    return name
 
def make_combo_key(a, b):
    a, b = normalize_name(a), normalize_name(b)
    return "-".join(sorted([a, b]))

def discretize(x):
    if x > TH:
        return 1
    elif x < -TH:
        return -1
    else:
        return 0

def load_flux_states(fp):
    if not os.path.exists(fp):
        return {}
    df = pd.read_csv(fp)
    if "Solution_Status" in df.columns:
        df = df[df["Solution_Status"] == "optimal"]
    df["Reaction"] = df["Reaction"].astype(str)
    df["ΔFlux"] = pd.to_numeric(df["ΔFlux"], errors="coerce").fillna(0.0)
    df["state"] = df["ΔFlux"].apply(discretize)
    return dict(zip(df["Reaction"], df["state"]))

# =========================
# Compatible with all filename formats
# =========================
def parse_combo_name(filename):
    name = filename.replace("gimme_flux_", "").replace(".csv", "")
    if "__VS__" in name:
        parts = name.split("__VS__")
    else:
        parts = name.split("-")
    if len(parts) != 2:
        return None, None
    A, B = parts
    return normalize_name(A), normalize_name(B)

# =========================
# 1. Read combination files
# =========================
all_files = os.listdir(COMBO_DIR)
combo_files = []
invalid_files = []
for f in all_files:
    if not f.endswith(".csv"):
        continue
    if "-" not in f and "__VS__" not in f:
        invalid_files.append(f)
        continue
    combo_files.append(f)
print("✅ Valid combo files:", len(combo_files))
print("⚠️ Invalid files skipped:", len(invalid_files))

# =========================
# 2. Extract combinations
# =========================
combos = []
drug_set = set()
for f in combo_files:
    A, B = parse_combo_name(f)
    if A is None:
        print("⚠️ Skip:", f)
        continue
    combos.append((f, A, B))
    drug_set.update([A, B])
print("Total combos:", len(combos))

# =========================
# 3. Single drugs
# =========================
single_state = {}
all_reactions = set()
for d in drug_set:
    fp = os.path.join(SINGLE_DIR, f"gimme_flux_{d}.csv")
    state = load_flux_states(fp)
    single_state[d] = state
    all_reactions.update(state.keys())

# =========================
# 4. combo (with cache + reaction completion)
# =========================
combo_state_cache = {}
for f, _, _ in combos:
    fp = os.path.join(COMBO_DIR, f)
    state = load_flux_states(fp)
    combo_state_cache[f] = state
    all_reactions.update(state.keys())
all_reactions = sorted(list(all_reactions))

# =========================
# 5. Build A2C / B2C features
# =========================
rows = []
for file_name, A, B in combos:
    print("Processing:", file_name)
    combo_key = make_combo_key(A, B)
    row = {"Combination": combo_key}

    A_state = single_state.get(A, {})
    B_state = single_state.get(B, {})
    C_state = combo_state_cache[file_name]

    for rxn in all_reactions:
        a = A_state.get(rxn, 0)
        b = B_state.get(rxn, 0)
        c = C_state.get(rxn, 0)

        # Keep full information (most important!)
        row[f"{rxn}_A"]   = a
        row[f"{rxn}_B"]   = b
        row[f"{rxn}_C"]   = c
        row[f"{rxn}_A2C"] = c - a
        row[f"{rxn}_B2C"] = c - b

    rows.append(row)

# =========================
# 6. Label handling (auto-aligned)
# =========================
label_df = pd.read_csv(LABEL_FILE, encoding="gbk")
label_df["Drug 1"] = label_df["Drug 1"].apply(normalize_name)
label_df["Drug 2"] = label_df["Drug 2"].apply(normalize_name)
label_df["Combination"] = label_df.apply(lambda x: make_combo_key(x["Drug 1"], x["Drug 2"]), axis=1)
label_df = label_df.rename(columns={"Experimental Interaction Score": "Label"})
label_df = label_df[["Combination", "Label"]]

# =========================
# 7. Merge
# =========================
out_df = pd.DataFrame(rows)
out_df = out_df.merge(label_df, on="Combination", how="left")

# =========================
# 8. Check
# =========================
missing = out_df["Label"].isna().sum()
print("\nMissing labels:", missing)
if missing > 0:
    print(out_df[out_df["Label"].isna()]["Combination"])

# =========================
# 9. Save
# =========================
out_df.to_csv(OUT_FILE, index=False)
print("\n✅ Saved:", OUT_FILE)
print("Shape:", out_df.shape)

✅ Valid combo files: 58
⚠️ Invalid files skipped: 0
Total combos: 58
Processing: gimme_flux_AMIKACIN-CEFOXITIN.csv
Processing: gimme_flux_AMIKACIN-GENTAMICIN.csv
Processing: gimme_flux_AMIKACIN-TETRACYCLINE.csv
Processing: gimme_flux_AMIKACIN-VANCOMYCIN.csv
Processing: gimme_flux_CEFOXITIN-CHLORAMPHENICOL.csv
Processing: gimme_flux_CEFOXITIN-CIPROFLOXACIN.csv
Processing: gimme_flux_CEFOXITIN-CLARITHROMYCIN.csv
Processing: gimme_flux_CEFOXITIN-ERYTHROMYCIN.csv
Processing: gimme_flux_CEFOXITIN-GENTAMICIN.csv
Processing: gimme_flux_CEFOXITIN-LEVOFLOXACIN.csv
Processing: gimme_flux_CEFOXITIN-NALIDIXICACID.csv
Processing: gimme_flux_CEFOXITIN-NITROFURANTOIN.csv
Processing: gimme_flux_CEFOXITIN-OXACILLIN.csv
Processing: gimme_flux_CEFOXITIN-RIFAMPICIN.csv
Processing: gimme_flux_CEFOXITIN-SPECTINOMYCIN.csv
Processing: gimme_flux_CEFOXITIN-TETRACYCLINE.csv
Processing: gimme_flux_CEFOXITIN-TOBRAMYCIN.csv
Processing: gimme_flux_CEFOXITIN-TRIMETHOPRIM.csv
Processing: gimme_flux_CEFOXITIN-VANCOMYC